## Encoder stack

In [2]:
import torch
import torch.nn as nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()

        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn_out,_ = self.attn(x,x,x)

        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)

        x = self.norm2(x + ffn_out)

        return x


class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, heads, d_ff):

        super().__init__()

        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, heads, d_ff)
             for _ in range(num_layers)]
        )

    def forward(self,x):

        for layer in self.layers:
            x = layer(x)

        return x


# TEST
x = torch.rand(2,5,16)

encoder = Encoder(2,16,4,64)

out = encoder(x)

print("Input:",x.shape)
print("Output:",out.shape)
print(out[0])

Input: torch.Size([2, 5, 16])
Output: torch.Size([2, 5, 16])
tensor([[ 1.2055, -1.0204, -0.8280, -0.1560,  0.7812,  0.9901, -1.9110,  1.1655,
         -1.4594,  0.2109, -0.0423,  0.4817,  0.2234, -0.1693,  1.5998, -1.0716],
        [-1.1626,  0.4406,  0.5240,  0.2024, -0.5120,  0.2192, -1.9350,  0.5612,
          0.3377,  0.2906, -1.0033,  1.5513,  0.5141, -1.9612,  1.2652,  0.6680],
        [ 1.3281,  0.1389,  0.7884, -2.1381,  1.0087,  0.1555,  0.0405, -0.0215,
         -0.3859, -0.1884, -1.4919,  1.5526, -0.5640, -1.4045,  0.8829,  0.2986],
        [ 1.5404, -0.9014,  0.7427,  0.1779,  0.0039,  0.0231, -1.8544,  1.2554,
          0.5644,  0.0651, -0.0106,  1.8742, -1.0441, -1.1585, -0.3831, -0.8950],
        [ 2.1947, -1.0075,  0.3122,  0.2080, -0.4906, -1.2219, -0.6830, -0.5154,
         -0.8053, -1.2025,  1.1751,  1.8748,  0.2141, -0.3549, -0.2694,  0.5718]],
       grad_fn=<SelectBackward0>)
